# 01 · Entendimento de negócio e descrição dos dados

**CRISP-DM fases 1 e 2.1–2.2** · Meridiano Imobiliária

---

## O problema, em uma frase

A Meridiano administra uma carteira de **2.930 imóveis residenciais** transacionados
em Ames, Iowa, entre 2006 e 2010. Hoje todos recebem a mesma política de preço, de
reforma e de anúncio. A diretoria quer saber **como esses imóveis se organizam
naturalmente** e tem orçamento para, no máximo, **cinco linhas de produto** com ação
diferenciada.

## O que faz deste um problema não supervisionado

Não existe rótulo. Ninguém nunca marcou qual imóvel é de "entrada" e qual é de "alto
padrão" — essa é justamente a pergunta. O preço de venda existe na base, mas é
**rótulo reservado**: usá-lo na modelagem transformaria a tarefa em regressão e
devolveria uma tabela de preço, que a diretoria já tem.

> **Critério eliminatório da disciplina.** `preco_venda` não entra em nenhuma etapa
> de modelagem. Ele reaparece uma única vez, no notebook 06, para responder *quanto
> vale cada segmento* — depois que os segmentos já estão definidos.

In [ ]:
import sys, warnings
from pathlib import Path

# O notebook roda a partir de `notebooks/`; a raiz do projeto entra no path
# para que `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from src import config
from src.utils.io import configurar_estilo

configurar_estilo()
print(f"Projeto em {RAIZ}")

## 2.1 · Coleta dos dados iniciais

A base vem do endereço oficial do artigo que a publicou. O download é direto, sem
cadastro, e a carga é **idempotente**: reexecutar não baixa de novo nem duplica dado.

> De Cock, D. (2011). *Ames, Iowa: Alternative to the Boston Housing Data as an End
> of Semester Regression Project.* Journal of Statistics Education, 19(3).

In [ ]:
from src.data.ingestao import baixar, carregar_bruto, inventario_sentinelas

manifesto = baixar()
pd.Series(manifesto).to_frame("valor")

In [ ]:
bruto = carregar_bruto()
print(f"Dimensões: {bruto.shape[0]} imóveis × {bruto.shape[1]} colunas")
print(f"Período: {int(bruto['Yr Sold'].min())} a {int(bruto['Yr Sold'].max())}")
bruto.iloc[:5, :12]

## 2.2 · Descrição dos dados

### O grão

Cada linha é **uma transação de um imóvel residencial**, identificada por `PID`
(*Parcel Identification*). A conferência abaixo prova que a chave identifica de fato
cada registro — sem isso, qualquer agregação estaria somando o mesmo imóvel duas
vezes.

In [ ]:
from src.data import qualidade as q

pd.Series(q.diagnostico_duplicidade(bruto, "PID")).to_frame("valor")

### Perfil das colunas

As colunas ordenadas por percentual de ausência. Repare no bloco do topo: a ausência
não está espalhada, está concentrada em colunas que descrevem itens que o imóvel pode
simplesmente **não ter**.

In [ ]:
perfil = q.perfil_colunas(bruto)
perfil[perfil["nulos"] > 0][["tipo", "nulos", "pct_nulos", "distintos"]]

### O achado que muda a leitura da base

O arquivo grava `NA` para "não se aplica" e a palavra **`None`** como categoria
legítima de `Mas Vnr Type` — "sem revestimento em alvenaria aparente". A leitura
padrão do `pandas` converte as duas em nulo.

O efeito: 1.752 imóveis **descritos** viram 1.752 imóveis **desconhecidos**, e a
coluna aparenta 60,6% de ausência quando a ausência real é de 0,8%.

In [ ]:
inventario_sentinelas().head(20)

### Colunas sem poder de separação

Três colunas concentram mais de 99,5% dos registros em um único valor. Variância
quase nula não separa grupo nenhum — e ainda gera divisão por zero na padronização.

In [ ]:
q.colunas_constantes(bruto)

### Regras de plausibilidade do domínio

Checagens específicas do mercado imobiliário. Regra sem violação também aparece: é a
evidência de que a checagem foi feita, e não presumida.

In [ ]:
from src.data import dicionario as dic

renomeado = bruto.rename(columns=dic.RENOMEAR)
q.valores_implausiveis(renomeado)

O item **"garagem construída depois da venda"** é um erro de digitação real:
um registro traz `Garage Yr Blt = 2207` para um imóvel vendido em 2007. Sem a
correção, a idade da garagem fica negativa em 200 anos e domina qualquer
padronização.

Os **5 imóveis acima de 4.000 pés²** são removidos por recomendação explícita do
autor da base (De Cock, 2011, §4): são vendas parciais de imóveis atípicos. Decisão
citável, e não arbitrária.

In [ ]:
acima = renomeado[renomeado["area_habitavel"] > 4000]
acima[["pid", "bairro", "area_habitavel", "qualidade_geral", "condicao_venda", "preco_venda"]]

## O Canvas do Problema, preenchido

| Campo | Conteúdo |
|---|---|
| **1. Contexto de negócio** | Meridiano Imobiliária: compra, requalifica e revende imóveis residenciais, além de intermediar vendas de terceiros. Quem decide é a Diretoria Comercial, com apoio da Gerência de Ativos. |
| **2. A dor** | Política única de preço, reforma e anúncio para 2.925 imóveis muito diferentes entre si. Reforma-se o que não valoriza e anuncia-se pelo canal errado. |
| **3. Objetivo de negócio** | Definir linhas de produto com política própria de precificação, teto de reforma e canal de venda. |
| **4. Critério de sucesso do negócio** | Reduzir o tempo médio de giro do estoque e elevar o retorno sobre o capital investido em reforma, medidos por linha de produto. |
| **5. Meta de mineração** | Agrupamento (segmentação) + regras de associação sobre a configuração dos imóveis. |
| **6. Critério de sucesso técnico** | Silhueta ≥ 0,15; entre 2 e 5 grupos; ruído ≤ 10%; estabilidade ARI ≥ 0,75 sob reamostragem. |
| **7. Entidade de análise** | Um imóvel residencial transacionado (`pid`). |
| **8. Fonte e volume** | Ames Housing, De Cock (2011). 2.930 × 82, 2006–2010. Uso educacional com citação obrigatória. |
| **9. Perguntas orientadas a dados** | Ver abaixo. |
| **10. Restrições e riscos** | Orçamento para no máximo 5 ações; base histórica de 2006–2010, que inclui a crise imobiliária; ausência de tempo de permanência em estoque. |
| **11. Ação esperada por grupo** | Política de precificação, teto de investimento em reforma, canal de venda e meta de giro — uma por linha de produto. |

### As perguntas SMART

1. Quantos grupos distintos emergem entre os 2.925 imóveis quando se consideram porte,
   configuração, padrão construtivo e idade, com silhueta ≥ 0,15 e no máximo 5 grupos?
2. Qual grupo concentra mais de 40% da receita transacionada no período de 2006 a 2010?
3. Quais combinações de até três atributos construtivos ocorrem juntas com suporte ≥ 10%
   e lift ≥ 1,30, e quais delas valem apenas dentro de um grupo específico?
4. Que percentual dos imóveis é classificado como ruído por um DBSCAN calibrado com
   `minPts = 2 × d`, e qual é o perfil desses imóveis?
5. A segmentação obtida é distinguível de uma simples faixa de preço, medida por um
   índice de Rand ajustado abaixo de 0,40 contra os quartis de `preco_venda`?

## Próximo passo

O notebook [`02-eda-fluxo-8-fases`](02-eda-fluxo-8-fases.ipynb) percorre as oito fases
da análise exploratória sobre a base curada.